# Case study: fifty years of state unemployment, segmented jointly

The table is the monthly unemployment rate for each US state and
the District of Columbia, January 1976 to September 2026: 609
months by 51 series, from the Bureau of Labor Statistics' Local
Area Unemployment Statistics via FRED (the corpus panel
`benchmarks/corpus/data/crosscat/state-unemployment`). The
question is where the panel's behaviour changes regime — the
months after which the states' rates are jointly distributed
differently from the months before.

The [changeforest detector](../methodology/changeforest.ipynb)
answers that question directly: it segments the measures jointly
along the date axis and states the splits. This study runs it on
the panel, reads the segmentation segment by segment, and then
runs the breaks pass on the same table for contrast.

In [ ]:
import datetime as dt

import polars as pl

import onceover

panel = pl.read_csv(
    "../../../benchmarks/corpus/data/crosscat/state-unemployment/data.csv"
).with_columns(pl.col("date").str.to_date())
print("panel:", panel.shape)

report = onceover.profile(panel, detectors=["changeforest"])
entry = report.coverage["changeforest"]
print(entry["status"], "-", entry["detail"])
changes = report.select(type="distribution_change")
print()
print(changes[0]["claim"])
print()
for f in changes:
    m = f["evidence"][0]["metrics"]
    print(f"split at {m['breakpoint']}  gain {m['gain']:.1f}  p {m['p_value']}")

Fifteen splits clear the 0.01 bar; ten are stated, the largest
by gain, and the coverage entry says so. It also records the
bound that shapes this run: 51 series exceed the 32-measure cap,
so the segmentation runs on the first 32 state columns in the
file's order, and the entry names the 19 set aside — North
Carolina through Wyoming and DC. Every number below is computed
on the 32 segmented series.

One split dominates. The split at March 2001 carries gain
409.7, twice the next largest. The table on the next page
summarises the eleven segments the ten splits cut, so that
split can be read against its neighbours.

In [ ]:
m0 = changes[0]["evidence"][0]["metrics"]
segmented = m0["measures"]
print(len(segmented), "segmented measures")
long = panel.select(["date"] + segmented).unpivot(
    index="date", variable_name="state", value_name="rate")
table = []
for s in m0["segments"]:
    rows = long.filter(
        (pl.col("date") >= dt.date.fromisoformat(s["from"]))
        & (pl.col("date") <= dt.date.fromisoformat(s["to"])))
    spread = rows.group_by("date").agg(pl.col("rate").std().alias("sd"))
    table.append({
        "segment": s["segment"],
        "from": s["from"],
        "to": s["to"],
        "months": s["rows"],
        "mean rate": round(rows["rate"].mean(), 2),
        "highest state rate": round(rows["rate"].max(), 1),
        "mean cross-state sd": round(spread["sd"].mean(), 2),
    })
print(pl.DataFrame(table))

In [ ]:
import altair as alt

monthly_mean = (long.group_by("date")
                .agg(pl.col("rate").mean().alias("mean_rate"))
                .sort("date"))
split_dates = pl.DataFrame(
    {"date": [dt.date.fromisoformat(s["from"]) for s in m0["segments"][1:]]})
line = (alt.Chart(monthly_mean).mark_line()
        .encode(x=alt.X("date:T", title="Month"),
                y=alt.Y("mean_rate:Q", title="Mean unemployment rate (%)")))
rules = (alt.Chart(split_dates).mark_rule(color="#b2182b", opacity=0.55)
         .encode(x="date:T"))
(line + rules).properties(
    title="Mean rate across the 32 segmented state series, "
          "with the ten stated splits",
    width=640, height=260)

Read chronologically, the segmentation follows the panel's
regimes. The split at February 1982 opens the highest segment in
the table: February 1982 to June 1985 averages 8.24% across the
32 series, and the two segments covering 1982–1988 carry the
widest cross-state spreads (2.13 and 2.23). In those years the
states differed most from each other — the early-1980s recessions
raised unemployment far more in manufacturing and energy states
than elsewhere. The split at July 1985 closes that block; the
segments after it step down through the 1990s to the panel's
quietest stretch, February 1998 to February 2001, with a mean of
4.10% and no state above 6.5%.

The March 2001 split, the largest by gain, does not sit on a
level jump: the pooled mean rate is 4.07% in February 2001 and
4.14% in March. What changes is the segment that follows. Its 40
months average 5.19% with the narrowest cross-state spread in
the table, 0.89 — the early-2000s rise was shared broadly across
states rather than concentrated in a few, and a change of that
shape is what the joint segmentation scores highest. The pooled
mean at the August 2007 split is nearly as flat (4.36% in July,
4.42% in August); the segment it opens runs 78 months at a mean
of 7.42%, the financial-crisis years and the slow decline after
them, held as one segment because the panel's distribution stays
in that regime throughout.

The April 2020 split is the one place where the level itself
jumps: the pooled mean goes from 4.15% in March 2020 to 13.99%
in April. Its gain, 100.8, is mid-sized, because the segment it
opens settles back: the final segment's mean, 4.53%, matches
the segment before the split (4.50%), while its highest state
rate is 30.5% against 8.8% — Nevada's rate in the spring of
2020. The last segment differs from its predecessor in its
tail, not its centre, and the joint statement records the
boundary even though the excursion itself lasted months.

## What the breaks pass states on the same panel

The breaks pass reads one series at a time. It scans 16 of the
51 series at the standard budget — the ones its prescore ranks
highest — and states what it finds series by series:

In [ ]:
from collections import Counter

breaks_report = onceover.profile(panel, detectors=["breaks"])
print(Counter(f["type"] for f in breaks_report.findings))
b_entry = breaks_report.coverage["breaks"]
print(b_entry["status"], "-", b_entry["detail"][:300], "...")
for wanted in ("level_shift", "transient_excursion"):
    f = next(f for f in breaks_report.findings if f["type"] == wanted)
    print()
    print(f["claim"])

The two passes divide the panel between them. Breaks states
84 findings: level shifts located per state, with a range —
Alaska's early-1980s shift is placed between June 1981 and
October 1982, best estimate November 1981 — and the April 2020
spike stated as a transient excursion in state after state —
16 of its 20 excursion findings start in April or May 2020. Changeforest states
the same years as panel-level boundaries: two splits around
the early-1980s block, one split at April 2020. Neither account
replaces the other. The per-state statements carry the fitted
quantities (segment means, holdout error reduction); the joint
statement carries the segmentation of the whole panel,
including the March 2001 boundary, where no state's level,
trend, or variance carries the change on its own.

Three limits apply to the joint account here. The segmentation
ran on 32 of the 51 series; the 19 set aside include several
large states, and a segmentation over a different 32 could
place some splits a month or two away. Five of the fifteen
significant splits are not stated — the findings name the ten
largest by gain, and the coverage entry gives the total, but
the five unstated positions are not in the report. And the
statements are model outputs: `report.verify` marks each one
`not_checkable`, so the segment table above, computed from the
panel itself, is the check a reader has.